In [4]:
from pathlib import Path
import pandas as pd

VENTAS_PATH=Path("data/ventas.csv")
df=pd.read_csv(VENTAS_PATH)

In [ ]:
df.shape


(150, 6)

In [9]:
df.dtypes

fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

In [10]:
df.head()

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027


In [11]:
df.isna().sum()

fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64

In [ ]:
##Observamos que en las columnas unidades y precio_unidades hay y 2 y 1 filas vacías respectivamente## 

In [12]:
df["unidades"].value_counts()

unidades
20    9
9     9
22    8
16    8
2     8
5     8
4     8
12    8
10    7
24    7
15    6
23    6
13    6
3     6
7     5
17    5
11    5
19    5
1     4
21    4
25    3
6     3
8     3
0     2
18    2
na    1
-2    1
14    1
Name: count, dtype: int64

In [13]:
df["precio_unitario"].value_counts()

precio_unitario
 12.50     28
 120.00    28
 19.50     27
 85.00     23
 32.00     22
 12.48      2
 88.96      1
 88.04      1
 83.72      1
 83.74      1
 0.00       1
 83.38      1
 19.59      1
 31.79      1
 19.00      1
 13.08      1
 33.50      1
 30.73      1
-1.00       1
 84.30      1
 32.84      1
 33.53      1
-5.00       1
 32.31      1
-32.00      1
Name: count, dtype: int64

In [17]:
df["region"].value_counts()

region
Norte    46
Este     39
Sur      37
Oeste    28
Name: count, dtype: int64

In [18]:
df["producto"].value_counts()

producto
Cable Kit    33
Hub IoT      30
Sensor A     30
Sensor B     29
Gateway      28
Name: count, dtype: int64

Conclusiones:
-Debemos examinar los datos de "unidades" y "precio_unitario" ya que contiene filas vacías y con valor negativo y 0 o nulo., lo cual no tiene sentido
-Me ha llamado la atención tambien que la variable "unidades" haya sido determinada por pandas como un str(debido a que hay una fila que contiene nan)
-Tambien considero que crear una nnueva columna llamada precio_total=unidades*precio unitario sería de gran ayuda


In [19]:
# 1. Forzamos la conversión a números en unas columnas temporales
# errors='coerce' transforma la palabra "nan" o cualquier texto inválido en un NaN real
unidades=pd.to_numeric(df["unidades"],errors="coerce")
precio= pd.to_numeric(df['precio_unitario'], errors='coerce')

#Defininimos que consideramos un ERROR para cada columna
error_unidades = unidades.isna() | (unidades <= 0)
error_precio = precio.isna() | (precio <= 0)

# Filtramos el DataFrame original donde ocurra CUALQUIERA de los dos errores
df_errores = df[error_unidades | error_precio]

#Creamos la lista que pedías con la posición original y el cotennido
lista_errores = [
    {"posicion_original": indice, "contenido": fila.to_dict()} 
    for indice, fila in df_errores.iterrows()
]


In [20]:
for fila in lista_errores:
    print(fila)

{'posicion_original': 3, 'contenido': {'fecha': '2026-02-05', 'region': 'Norte', 'producto': 'Gateway', 'unidades': '0', 'precio_unitario': 120.0, 'cliente_id': 'C010'}}
{'posicion_original': 41, 'contenido': {'fecha': '2026-02-09', 'region': 'Oeste', 'producto': 'Cable Kit', 'unidades': '4', 'precio_unitario': 0.0, 'cliente_id': 'C015'}}
{'posicion_original': 54, 'contenido': {'fecha': '2026-02-01', 'region': 'Sur', 'producto': 'Sensor B', 'unidades': nan, 'precio_unitario': 32.0, 'cliente_id': 'C004'}}
{'posicion_original': 63, 'contenido': {'fecha': '2026-02-11', 'region': 'Norte', 'producto': 'Sensor A', 'unidades': nan, 'precio_unitario': 19.5, 'cliente_id': 'C018'}}
{'posicion_original': 66, 'contenido': {'fecha': '2026-02-17', 'region': 'Oeste', 'producto': 'Hub IoT', 'unidades': '5', 'precio_unitario': nan, 'cliente_id': 'C025'}}
{'posicion_original': 84, 'contenido': {'fecha': '2026-02-15', 'region': 'Este', 'producto': 'Gateway', 'unidades': 'na', 'precio_unitario': 120.0, 'c

In [21]:
df_errores=pd.DataFrame(lista_errores)

In [29]:
def Validar_ventas(df_raw) -> tuple[pd.DataFrame,pd.DataFrame]:
    unidades=pd.to_numeric(df_raw["unidades"],errors="coerce")
    precio= pd.to_numeric(df_raw['precio_unitario'], errors='coerce')

    error_unidades= (unidades <=0) | (unidades.isna())
    error_precio = precio.isna() | (precio <= 0)

    df_errores=df_raw[error_precio | error_unidades].copy()

    validos=~(error_unidades|error_precio)
    df_validos=df_raw[validos].copy()

    df_validos["unidades"] = unidades[validos].astype(int)
    df_validos["precio_unitario"] = precio[validos]

    df_validos["importe"]=df_validos["unidades"]*df_validos["precio_unitario"]

    return (df_validos,df_errores)


In [30]:
df_validos , df_errores = Validar_ventas(df) 
df_validos.head()

,fecha,region,producto,unidades,precio_unitario,cliente_id,importe
0,2026-01-09,Sur,Cable Kit,7.0,12.50,C005,87.50
1,2026-03-14,Oeste,Gateway,25.0,120.00,C032,3000.00
2,2026-01-26,Oeste,Hub IoT,1.0,88.96,C016,88.96
4,2026-02-07,Norte,Hub IoT,15.0,85.00,C027,1275.00
5,2026-01-11,Este,Cable Kit,22.0,12.50,C028,275.00


In [48]:
fact_por_region=df_validos.groupby("region")["importe"].sum().sort_values(ascending=False)
print(fact_por_region)

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64


In [ ]:
fact_por_product=df_validos.groupby("producto")["importe"].sum().sort_values(ascending=False)
fact_por_product.head(3)

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64

In [56]:
ventas_por_product=df_validos.groupby("producto")["unidades"].sum().sort_values(ascending=False)
ventas_por_product.head(3)

producto
Cable Kit    391.0
Hub IoT      389.0
Sensor B     359.0
Name: unidades, dtype: float64

In [68]:
compras_por_cliente=df_validos["cliente_id"].value_counts()  

In [70]:
cliente_habitual=compras_por_cliente[compras_por_cliente > 1]
print(cliente_habitual)

cliente_id
C027    9
C032    8
C004    8
C035    7
C013    6
C009    6
C005    5
C022    5
C038    5
C010    4
C018    4
C002    4
C040    4
C025    4
C015    4
C024    4
C023    4
C034    4
C037    4
C016    3
C003    3
C007    3
C011    3
C029    3
C033    3
C030    3
C028    2
C036    2
C021    2
C039    2
C031    2
C001    2
C020    2
C017    2
Name: count, dtype: int64
